# slide2pptx: two-level box detection (U-Net + pair CNN)

**Runtime → Change runtime type → T4 GPU.** No dataset needed: slides are generated on the
fly by `boxseg/synth.py`. Checkpoints go to Drive. See `docs/two_level.md` for the idea.

* Level 1: U-Net predicts *inside* and *edge* maps → every closed box, class-agnostic.
* Level 2: a small CNN decides for each nearby pair of boxes whether they belong to one group.

In [ ]:
# 1. Settings
REPO_URL  = "https://github.com/C00p1r/slide2pptx.git"
BRANCH    = "claude/project-thread-jkuii6"
DRIVE_DIR = "/content/drive/MyDrive/slide2pptx"
OUT       = f"{DRIVE_DIR}/runs/boxseg"
SIZE      = "512x288"      # generated slide size (16:9)
SEG_STEPS, GROUP_STEPS = 8000, 4000

In [ ]:
# 2. Mount Drive, clone the branch, install
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/slide2pptx'):
    !git clone -b {BRANCH} {REPO_URL} /content/slide2pptx
else:
    !git -C /content/slide2pptx fetch origin {BRANCH} && git -C /content/slide2pptx checkout {BRANCH} && git -C /content/slide2pptx pull
%cd /content/slide2pptx
!pip install -q opencv-python-headless
!apt-get -qq install -y fonts-noto-cjk fonts-liberation > /dev/null
!python -m boxseg.synth --n 6 --out /content/samples.png
from IPython.display import Image as Show; Show('/content/samples.png', width=900)

Level 1 (main experiment). On a T4 about 0.15 s/step at batch 16; data generation on the CPU workers is usually the bottleneck. Long jobs: run from the Colab terminal so a page disconnect doesn't kill them.

In [ ]:
!python -m boxseg.train seg --steps {SEG_STEPS} --batch 16 --size {SIZE} --workers 2 --out {OUT}

In [ ]:
# Level 2
!python -m boxseg.train group --steps {GROUP_STEPS} --batch 64 --size {SIZE} --workers 2 --out {OUT}

In [ ]:
# Evaluate on 300 fresh synthetic scenes: U-Net vs classical contours, per-style recall, grouping
!python -m boxseg.evaluate --seg {OUT}/seg.pt --group {OUT}/group.pt --n 300

**Held-out style experiment** (the key question: does it learn geometry rather than style?).
Train a second U-Net that never sees hand-drawn outlines or sticky notes, then test only on
boxes with those styles. If recall stays close to the full model, the network generalises.

In [ ]:
!python -m boxseg.train seg --steps {SEG_STEPS} --batch 16 --size {SIZE} --workers 2 --out {OUT} --name seg_no_hand --exclude-styles hand sticky
!python -m boxseg.evaluate --seg {OUT}/seg_no_hand.pt --n 300 --only-styles hand sticky --report {OUT}/heldout_no_hand.md
!python -m boxseg.evaluate --seg {OUT}/seg.pt          --n 300 --only-styles hand sticky --report {OUT}/heldout_full.md

In [ ]:
# Real NotebookLM slides (same images as the YOLO tests)
REAL = f"{DRIVE_DIR}/data/real/images"
!python -m boxseg.predict {OUT}/seg.pt {REAL} --group {OUT}/group.pt --out {OUT}/pred_real --baseline